In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 88.7 MB/s eta 0:00:00


In [2]:
import faiss
import pandas as pd
import numpy as np
import gc

paths = [
    '/kaggle/input/notebooks/fati22/embedding-images-phase1-clip/chunk_1.parquet',
    '/kaggle/input/notebooks/agnes15/embedding-images-phase2-clip/chunk_2.parquet',
    '/kaggle/input/notebooks/agnes15/embedding-images-phase3-clip/chunk_3.parquet',
    '/kaggle/input/notebooks/drivecollab0/embedding-images-phase4-clip/chunk_4.parquet',
    '/kaggle/input/notebooks/drivecollab0/embedding-images-phase5-clip/chunk_5.parquet',
    '/kaggle/input/notebooks/drivecollab0/embedding-images-phase6-clip/chunk_6.parquet',
    '/kaggle/input/notebooks/fati567/embedding-images-phase7-clip/chunk_7.parquet',
    '/kaggle/input/notebooks/agnes15/embedding-images-phase8-clip/chunk_8.parquet',
    '/kaggle/input/notebooks/agnes15/embedding-images-phase9-clip/chunk_9.parquet',
    '/kaggle/input/notebooks/agnes15/embedding-images-phase10-clip/chunk_10.parquet',
    '/kaggle/input/notebooks/drivecollab0/embedding-images-phase11-clip/chunk_11.parquet',
    '/kaggle/input/notebooks/drivecollab0/embedding-images-phase12-clip/chunk_12.parquet',
    '/kaggle/input/notebooks/drivecollab0/embedding-images-phase13-clip/chunk_13.parquet',
    '/kaggle/input/notebooks/fati567/embedding-images-phase14-clip/chunk_14.parquet'
]


In [3]:
d_img  = 1024
M_hnsw = 32
ef     = 200

def create_index(dim):
    core_index = faiss.IndexHNSWFlat(dim, M_hnsw)
    core_index.hnsw.efConstruction = ef
    final_index = faiss.IndexIDMap2(core_index)
    return final_index

In [4]:
index_img = create_index(d_img)
for file_path in paths:
    df      = pd.read_parquet(file_path)
    ids     = df['ID_Product'].values.astype('int64')
    img_vecs = np.vstack(df['Image_Embedding'].values).astype('float32')
    index_img.add_with_ids(img_vecs, ids)
    del df, ids, img_vecs#,txt_vecs
    gc.collect()
faiss.write_index(index_img, "tokopedia_img_CLIP_hnsw.faiss")